# O que muda ao longo do tempo dentro de cada tipo de anúncio

Este caderno responde: **dentro do mesmo tipo de anúncio (fuga, aluguel, venda, ama de leite), o que muda de um período legal para outro?**
Comparar períodos misturando os tipos engana, porque a composição dos tipos muda de período para período. Aqui cada tipo é lido sozinho.
Para cada tipo e período mostramos (a) as **10 palavras que mais distinguem aquele período dos outros do mesmo tipo** e (b) alguns **atributos** dos anúncios.

Corpus `corp3`, campos `c6`. Os números dos comentários foram lidos da saída da própria execução;
se o banco mudar, **confira os números de novo**.

> **Cuidado ao ler:** os anúncios foram achados por **frases de busca**, não por sorteio. **Grupos com menos de 30 anúncios
> aparecem marcados com `⚠`** (e em itálico/negrito nas tabelas de palavras): as diferenças ali são frágeis.

## Como medimos as palavras que distinguem
Usamos o **log-odds ponderado com prior informativo** de Monroe, Colaresi e Quinn (2008). Para cada palavra, comparamos
quanto ela aparece no período contra quanto aparece nos **outros períodos do mesmo tipo**, e dividimos a diferença
(em log-odds) pelo seu desvio estimado, o que dá um **z-score**. O prior é proporcional à frequência da palavra no
conjunto inteiro do tipo, o que evita que palavras raras dominem. Só mostramos palavras com **z ≥ 1,96**, ordenadas por z;
a coluna entre parênteses é o número de vezes que a palavra aparece nos anúncios do período.

## Preparação: ler o corpus e os campos

In [1]:
import sys
from collections import Counter
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from hdb import config
from hdb.banco import BancoSupabase
from hdb.distintivas import palavras_distintivas

pd.set_option("display.width", 200)
banco = BancoSupabase()
db = banco.cliente      # lê o config/.env pelo módulo hdb

def ler_tudo(consulta, bloco=1000):
    """Executa a consulta em blocos de 1000 linhas e junta o resultado."""
    linhas, inicio = [], 0
    while True:
        parte = consulta().range(inicio, inicio + bloco - 1).execute().data
        linhas += parte
        if len(parte) < bloco:
            return pd.DataFrame(linhas)
        inicio += bloco

PERIODOS = ["antes de 1831", "1831–1850", "1850–1871", "1871–1888"]
def periodo_curto(p):
    return p if p.startswith("antes") else p.split(" ")[0]
MIN_N = 30

corpus = pd.DataFrame(banco.anuncios_do_corpus(config.VERSAO_CORPUS))
corpus = corpus[corpus["texto_padrao"].fillna("").str.strip() != ""]
# campos que o corpus não traz (nações, cores, ofícios, condição, recompensa, idade máxima), da view anuncios_completos
extra = ler_tudo(lambda: db.table("anuncios_completos").select("id,c_idade_max,c_nacoes,c_cores,c_oficios,c_recompensa")
                 .in_("versao", list(config.VERSOES_ANUNCIOS)).eq("c_versao", config.VERSAO_CAMPOS).order("id")).drop_duplicates("id")
df = corpus.merge(extra, on="id", how="left")
df["periodo"] = df["periodo_legal"].map(lambda p: periodo_curto(p) if isinstance(p, str) else None)
df = df[df["periodo"].isin(PERIODOS)].copy()
df["idade"] = df[["c_idade_min", "c_idade_max"]].astype(float).mean(axis=1)
tem = lambda s: s.map(lambda v: isinstance(v, list) and len(v) > 0)
print(f"{len(df)} anúncios no corpus {config.VERSAO_CORPUS} com período legal")
if len(df) != 2191:
    print(f"ATENÇÃO: os comentários deste caderno foram escritos para 2191 anúncios e agora são {len(df)}: "
          "confira os números dos textos antes de usar.")

TIPOS = [("fuga", "Fuga"), ("aluguel", "Aluguel"), ("venda", "Venda"), ("ama_de_leite", "Ama de leite")]
n_compra = int((df["c_tipo"] == "compra").sum())
if n_compra >= MIN_N:
    TIPOS.append(("compra", "Compra"))
print(f"Anúncios do tipo compra: {n_compra} ({'entra' if n_compra >= MIN_N else 'fica de fora: menos de 30'})")

def atributos(g, tipo):
    """Atributos de um grupo (um período dentro de um tipo)."""
    n = len(g)
    pct = lambda mask: round(100 * float(mask.sum()) / n, 1) if n else float("nan")
    linha = {"n": f"{n} ⚠" if n < MIN_N else str(n),
             "% feminino": pct(g["c_sexo"] == "f"),
             "% com idade": pct(g["idade"].notna()),
             "idade mediana": round(float(g["idade"].median()), 1) if g["idade"].notna().any() else float("nan"),
             "% com nação": pct(tem(g["c_nacoes"])),
             "% com ofício": pct(tem(g["c_oficios"])),
             "% com cor": pct(tem(g["c_cores"]))}
    if tipo == "fuga":
        linha["% com recompensa"] = pct(g["c_recompensa"].fillna(False).astype(bool))
    return linha

2191 anúncios no corpus corp3 com período legal
Anúncios do tipo compra: 36 (entra)


## 1. Quantos anúncios por tipo e período
Os grupos com `⚠` têm menos de 30 anúncios.

In [2]:
t1 = pd.crosstab(df["c_tipo"], df["periodo"]).reindex(columns=PERIODOS, fill_value=0)
t1["Total"] = t1.sum(axis=1)
t1.map(lambda v: f"{v} ⚠" if v < MIN_N else str(v))

periodo,antes de 1831,1831–1850,1850–1871,1871–1888,Total
c_tipo,,,,,
aluguel,18 ⚠,159,494,181,852
ama_de_leite,25 ⚠,71,111,30,237
compra,0 ⚠,17 ⚠,16 ⚠,3 ⚠,36
fuga,68,108,178,93,447
venda,41,210,286,82,619


**O que mostra:** são 447 fugas (68, 108, 178 e 93 por período), 852 aluguéis, 619 vendas, 237 amas de leite e 36 compras. As células com menos de 30 anúncios vêm marcadas: aluguel antes de 1831 (18), ama de leite antes de 1831 (25) e as compras em todos os períodos.

## 2. Palavras que distinguem cada período, dentro de cada tipo
Para cada tipo, as 10 palavras mais distintivas de cada período (z ≥ 1,96) em relação aos outros períodos **do mesmo tipo**.
Períodos com menos de 30 anúncios vêm com aviso: as palavras ali refletem poucos textos.

In [3]:
PALAVRAS = {}
for tipo, rotulo in TIPOS:
    sub = df[df["c_tipo"] == tipo]
    contagens = {p: Counter(w for t in sub[sub["periodo"] == p]["texto_padrao"] for w in t.split()) for p in PERIODOS}
    contagens = {p: c for p, c in contagens.items() if c}   # períodos sem anúncio do tipo ficam de fora
    ns = sub.groupby("periodo").size()
    dist = palavras_distintivas(contagens, n=10)
    PALAVRAS[tipo] = dist
    linhas = []
    for p in PERIODOS:
        n = int(ns.get(p, 0))
        pal = ", ".join(f"{w} ({k})" for w, z, k in dist.get(p, [])) or "(nenhuma com z ≥ 1,96)"
        if n == 0:
            continue
        linhas.append(f"| {p} | {'**⚠ ' + str(n) + '**' if n < MIN_N else n} | {('**⚠ poucos anúncios:** ' + pal) if n < MIN_N else pal} |")
    display(Markdown(f"### {rotulo}\n\n| Período | n | Palavras distintivas (n no período) |\n|---|---|---|\n" + "\n".join(linhas)))

### Fuga

| Período | n | Palavras distintivas (n no período) |
|---|---|---|
| antes de 1831 | 68 | alvicaras (31), souber (24), noticia (28), ordinaria (20), recebera (30), dele (22), queira (12), trabalho (10), boas (12), achar (9) |
| 1831–1850 | 108 | nome (95), meio (21), calcas (18), lenco (10), camisa (33), cabinda (13), nacao (57), corrente (63), recompensado (11), horas (10) |
| 1850–1871 | 178 | crioulo (35), rigor (18), sera (56), contra (32), pedestres (11), ugio (10), anda (25), protesta (25), lei (19), ter (22) |
| 1871–1888 | 93 | fugido (39), escravo (50), quantia (10), fazenda (22), pardo (22), cravo (7), gomes (7), escravos (8), dentes (19), fagio (7) |

### Aluguel

| Período | n | Palavras distintivas (n no período) |
|---|---|---|
| antes de 1831 | **⚠ 18** | **⚠ poucos anúncios:** quiser (5), procure (3), achara (2), detraz (2), mulher (2), vender (4), deste (2), cosa (2), dando (2), defronte (2) |
| 1831–1850 | 159 | alugar (36), precisa (51), dois (18), queira (9), corrente (7), ordinario (7), pretos (11), reis (6), alugao (18), roga (5) |
| 1850–1871 | 494 | aluga (418), engoma (50), mucama (17), alfandega (13), sabao (13), cozinha (61), lava (91), perfeitamente (14), cose (22), crioula (12) |
| 1871–1888 | 181 | fogao (14), general (9), lugao (8), sobrado (22), servicos (7), perfeita (15), escravas (5), bos (6), copeiro (13), forno (12) |

### Venda

| Período | n | Palavras distintivas (n no período) |
|---|---|---|
| antes de 1831 | 41 | quiser (17), comprar (11), pertender (11), sabe (12), procure (4), vendese (4), ainda (6), humilde (4), ourives (4), cosinhar (6) |
| 1831–1850 | 210 | alfaiate (12), cose (24), cozinhar (24), reis (10), continua (9), molestias (11), oficio (11), queira (8), vicios (23), cozinheiro (32) |
| 1850–1871 | 286 | pardo (28), perfeito (27), moca (18), fogao (21), forno (21), mucama (22), parda (23), peca (11), bonita (30), pagem (15) |
| 1871–1888 | 82 | informa (7), larga (5), vesdese (5), endese (4), tende (5), ade (3), indo (3), prio (3), piano (3), ancos (2) |

### Ama de leite

| Período | n | Palavras distintivas (n no período) |
|---|---|---|
| antes de 1831 | **⚠ 25** | **⚠ poucos anúncios:** quiser (14), alugar (7), amas (7), crianca (5), procure (4), queira (4), comprar (5), sabe (5), pertender (3), alguma (4) |
| 1831–1850 | 71 | cria (12), publico (6), vende (18) |
| 1850–1871 | 111 | ama (99) |
| 1871–1888 | 30 | sobrado (8), dias (6), lugao (3), seca (3) |

### Compra

| Período | n | Palavras distintivas (n no período) |
|---|---|---|
| 1831–1850 | **⚠ 17** | **⚠ poucos anúncios:** (nenhuma com z ≥ 1,96) |
| 1850–1871 | **⚠ 16** | **⚠ poucos anúncios:** (nenhuma com z ≥ 1,96) |
| 1871–1888 | **⚠ 3** | **⚠ poucos anúncios:** (nenhuma com z ≥ 1,96) |

**O que mostra (fuga, o recorte mais rico):** as palavras que mais distinguem cada período mudam de natureza. **Antes de 1831**, a fórmula do aviso: `alviçaras`, `souber`, `notícia`, `receberá`. **1831–1850**, a descrição do que a pessoa levava e de onde vinha: `camisa`, `calças`, `lenço`, `nação`, `Cabinda`. **1850–1871**, a linguagem de ameaça legal: `protesta`, `rigor`, `lei`, `contra` ("protesta-se com todo o rigor da lei contra quem a tiver acoutado"), e os `pedestres` (a quem se recorria para capturar). **1871–1888** (que agora inclui 1887), o escravo é nomeado como tal (`escravo`, `fugido`) e aparecem `fazenda` e `quantia`, o que combina com fugas de áreas rurais e recompensas em dinheiro. **São pistas, não prova**: o OCR deixa palavras partidas (`ugio`, `fagio`, `cravo`) e o último período tem 93 fugas. Nos demais tipos, as palavras distintivas de 1850–1871 vêm do próprio serviço doméstico (`engoma`, `mucama`, `lava`) e as de 1871–1888, de aluguéis de casas e fogões; vários grupos têm poucas palavras que passam do corte.

## 3. Atributos por período, dentro de cada tipo
% feminino, % com idade, idade mediana (só entre os que dizem a idade), % com nação, % com ofício, % com cor e, nas fugas,
% com recompensa. As porcentagens são sobre **todos** os anúncios do período naquele tipo. `n` com `⚠` = menos de 30 anúncios: não confie nas diferenças.

In [4]:
for tipo, rotulo in TIPOS:
    sub = df[df["c_tipo"] == tipo]
    linhas = {p: atributos(sub[sub["periodo"] == p], tipo) for p in PERIODOS if (sub["periodo"] == p).any()}
    display(Markdown(f"### {rotulo}"))
    display(pd.DataFrame(linhas).T.rename_axis("Período"))

### Fuga

,n,% feminino,% com idade,idade mediana,% com nação,% com ofício,% com cor,% com recompensa
Período,,,,,,,,
antes de 1831,68,23.5,26.5,17.5,63.2,5.9,45.6,66.2
1831–1850,108,12.0,46.3,20.8,50.0,16.7,71.3,65.7
1850–1871,178,14.6,31.5,24.5,27.5,12.9,62.4,47.2
1871–1888,93,15.1,28.0,29.0,0.0,6.5,40.9,25.8


### Aluguel

,n,% feminino,% com idade,idade mediana,% com nação,% com ofício,% com cor
Período,,,,,,,
antes de 1831,18 ⚠,50.0,0.0,NaN,0.0,16.7,94.4
1831–1850,159,51.6,7.5,16.0,0.0,27.7,85.5
1850–1871,494,59.5,3.4,18.0,0.0,27.3,88.7
1871–1888,181,67.4,4.4,19.0,0.6,24.9,81.8


### Venda

,n,% feminino,% com idade,idade mediana,% com nação,% com ofício,% com cor
Período,,,,,,,
antes de 1831,41,56.1,26.8,18.0,24.4,22.0,78.0
1831–1850,210,39.5,39.0,18.0,4.8,40.5,74.8
1850–1871,286,43.4,26.2,20.0,0.7,30.8,78.0
1871–1888,82,42.7,23.2,16.0,0.0,28.0,73.2


### Ama de leite

,n,% feminino,% com idade,idade mediana,% com nação,% com ofício,% com cor
Período,,,,,,,
antes de 1831,25 ⚠,92.0,8.0,21.5,4.0,44.0,48.0
1831–1850,71,83.1,16.9,18.5,0.0,78.9,49.3
1850–1871,111,89.2,9.0,20.0,0.0,71.2,43.2
1871–1888,30,93.3,3.3,22.0,0.0,70.0,23.3


### Compra

,n,% feminino,% com idade,idade mediana,% com nação,% com ofício,% com cor
Período,,,,,,,
1831–1850,17 ⚠,23.5,11.8,13.2,5.9,11.8,64.7
1850–1871,16 ⚠,50.0,12.5,9.5,0.0,6.2,75.0
1871–1888,3 ⚠,33.3,0.0,NaN,0.0,0.0,0.0


**O que se vê (fuga):** a **nação citada** cai de 63,2% antes de 1831 a 50,0%, 27,5% e 0,0% em 1871–1888; a **recompensa** vai de 66,2% a 65,7%, 47,2% e 25,8%; a **cor** é citada em 45,6%, 71,3%, 62,4% e 40,9%; e a **idade mediana** dos fugidos sobe de 17,5 para 20,8, 24,5 e 29,0 anos. Isso combina com anúncios cada vez menos descritivos e com pessoas mais velhas (lembrando que só 26% a 46% das fugas trazem idade). **No aluguel**, a parcela feminina sobe de 51,6% (1831–1850) para 59,5% e 67,4% (1871–1888). **Na venda**, ela é de 56,1% antes de 1831 e fica entre 39,5% e 43,4% depois. Na **ama de leite**, é de 83% a 93%. Mesmo dentro de cada tipo, vários grupos têm poucos anúncios, então as mudanças pequenas não valem como evidência.

## 4. O que isso diz e o que não diz

**O que sustenta:** dentro da **fuga**, o texto do anúncio muda de forma clara entre períodos (fórmula, descrição, ameaça legal), e a nação, a recompensa e a cor citadas **caem** ao longo do tempo, sem que isso dependa da mistura de tipos (aqui se compara só fuga com fuga).

**O que não sustenta:** a idade mediana só existe em uma parte pequena das fugas, e os tipos com menos de 30 anúncios (compra, aluguel e ama de leite em alguns períodos) não permitem nenhuma conclusão. As palavras distintivas dependem do OCR e do tamanho de cada grupo.